In [0]:
# ============================================================
# NOTEBOOK 02 — TRUSTED: PREPARACIÓN Y ANONIMIZACIÓN
# Vermont School - Sistema de Alerta Temprana 2025-26
# ============================================================
%pip install lxml

In [0]:
import pandas as pd
import numpy as np
from io import StringIO
from datetime import datetime
import hashlib, random, re, os

# Rutas del datalake
BRONZE  = "/Volumes/workspace/vermont/bronze"
TRUSTED = "/Volumes/workspace/vermont/trusted"

# Semilla fija para reproducibilidad
random.seed(42)
np.random.seed(42)

# Asignaturas académicas por grado
ASIG_7_8 = [
    'Integrated Science', 'Individuals and societies',
    'Educación Física', 'English', 'Lengua Castellana',
    'Mandarín', 'Mathematics', 'Financial Maths', 'ICT/STEM'
]
ASIG_9 = [
    'Life Science', 'Physical Science', 'Ciencias Políticas',
    'Educación Física', 'English', 'Lengua Castellana',
    'Mandarín', 'Research methodology',
    'Mathematics', 'Financial Maths', 'ICT/STEM'
]
PESOS = {
    'Primer Trimestre':  0.30,
    'Segundo Trimestre': 0.30,
    'Tercer Trimestre':  0.40
}

# Anonimización de secciones (aleatoria con seed fijo)
seccion_map = {}
for g in ['7', '8', '9']:
    l = ['SA', 'SB']; random.shuffle(l)
    seccion_map[f'{g}° A'] = f'G{g}{l[0]}'
    seccion_map[f'{g}° B'] = f'G{g}{l[1]}'

# Mapa de docentes
DOCENTES_MIDDLE = [
    "ACUÑA BARRAGAN JULIO CESAR",
    "RODRIGUEZ RODRIGUEZ ANDRES IGNACIO",
    "Huang Huang Wen Zhun",
    "Villa Uribe Veronica",
    "Duarte Alba Ingrid Paola",
    "Marsiglia Lopez Dalma Elizabeth",
    "Posada Penagos Alejandro",
    "POSADA RESTREPO DEISI YULIANA",
    "Gomez Mejia Didier Johan",
    "Velasco Hernandez Andres Felipe",
    "Villegas Cardona Fransua",
    "Lopez Vargas Francisco Antonio",
    "MOLINA PACHON LORENA ALEXANDRA",
    "TABOADA DURANGO JUAN ANTONIO",
    "Agudelo Rodriguez Elmer Ivar",
    "Arboleda Patiño Cipriano",
    "TORRES POSADA CRISTIAN CAMILO",
    "Zapata Mejía Luis Camilo",
    # Sin registros en F1/F2 pero docentes activos
    "Angela Gomez",
    "Carlos Garcia",
]
OTROS_ROLES = {
    "AGUDELO SERNA /RETIRADO SEBASTIÁN": "retirado",
    "Mercado Diaz Darwin Raul":          "retirado",
    "Brito Londoño Edwin":               "coordinador",
    "SELIGMAN DIEGO":                    "psicologo_hs",
}

# Asignar códigos P01-P20 aleatoriamente
codigos = [f'P{str(i).zfill(2)}' for i in range(1, 21)]
random.shuffle(codigos)
docente_map = {nombre: cod for nombre, cod in zip(DOCENTES_MIDDLE, codigos)}
for nombre, rol in OTROS_ROLES.items():
    docente_map[nombre] = rol

print("✓ Configuración cargada")
print(f"  Secciones anonimizadas: {seccion_map}")
print(f"  Docentes Middle mapeados: {len(DOCENTES_MIDDLE)}")
print(f"  Otros roles: {len(OTROS_ROLES)}")

In [0]:
# Leer notas desde Bronze
with open(f"{BRONZE}/report.xls", "r", encoding="utf-8", errors="replace") as f:
    raw = f.read()

tables = pd.read_html(StringIO(raw))
cursos_info = [
    ("7° A", ASIG_7_8), ("7° B", ASIG_7_8),
    ("8° A", ASIG_7_8), ("8° B", ASIG_7_8),
    ("9° A", ASIG_9),   ("9° B", ASIG_9)
]

registros = []
for tabla, (curso, asigs) in zip(tables, cursos_info):
    tabla.columns = [c[1] if isinstance(c, tuple) else c for c in tabla.columns]
    cn, cp = tabla.columns[0], tabla.columns[1]
    df_t = tabla[tabla[cp].isin(PESOS.keys())].copy()

    for est in df_t[cn].dropna().unique():
        de = df_t[df_t[cn] == est]
        rec = {'nombre_key': str(est).replace(',', '').strip(),
               'curso_orig': curso,
               'grado': curso[0]}
        notas_acum = {}
        for asig in asigs:
            if asig not in de.columns:
                notas_acum[asig] = np.nan
                continue
            np_, pt_ = 0, 0
            for per, peso in PESOS.items():
                fi = de[de[cp] == per]
                col_t = f"{asig}_T{list(PESOS.keys()).index(per)+1}"
                try:
                    v = float(fi[asig].values[0]) if not fi.empty else np.nan
                    rec[col_t] = v
                    if not np.isnan(v):
                        np_ += v * peso
                        pt_ += peso
                except:
                    rec[col_t] = np.nan
            notas_acum[asig] = round(np_ / pt_, 4) if pt_ > 0 else np.nan

        nv = [v for v in notas_acum.values() if not np.isnan(v)]
        b4 = sum(1 for v in nv if v < 4.0)
        nivel = ('sin_riesgo' if b4 == 0
                 else 'riesgo_recuperacion' if b4 <= 2
                 else 'riesgo_critico')

        rec.update({
            'n_asig_bajo_4':    b4,
            'nivel_riesgo':     nivel,
            'promedio_acumulado': round(np.nanmean(nv), 4) if nv else np.nan,
            'nota_min_acumulada': round(min(nv), 4) if nv else np.nan,
        })
        rec.update({f'nota_acum_{a}': v for a, v in notas_acum.items()})
        registros.append(rec)

df_notas = pd.DataFrame(registros)
print(f"✓ Notas procesadas: {len(df_notas)} estudiantes")
print(f"\nDistribución de riesgo:")
print(df_notas['nivel_riesgo'].value_counts().to_string())

In [0]:
# Leer asistencia, F1 y F2 desde Bronze
import re

# Asistencia
with open(f"{BRONZE}/exported.xls", "r", encoding="utf-8", errors="replace") as f:
    raw2 = f.read()
da = pd.read_html(StringIO(raw2))[0]
da = da[da['Estudiante'] != 'Total'].copy()
da['nombre_key'] = da['Estudiante'].apply(
    lambda x: str(x).replace(',', '').strip()
)
da = da.rename(columns={
    'Código':                     'codigo_matricula',
    'Matrícula':                  'seccion_raw',
    'Ausencia Clase':             'ausencia_clase',
    'Ausencia Clase con Excusa':  'ausencia_clase_excusa',
    'Ausencia Colegio':           'ausencia_colegio',
    'Ausencia Colegio con Excusa':'ausencia_colegio_excusa',
    'Retardo':                    'retardo',
    'Salida Temprano':            'salida_temprana',
    'Total':                      'total_ausencias',
})

# F1
df_f1 = pd.read_csv(f"{BRONZE}/Consolidado_seguimiento.xls",
                    sep=";", encoding="utf-8-sig")

def extraer_num_falta(texto):
    if pd.isna(texto): return None
    m = re.match(r'^\s*(\d+)\.', str(texto).strip())
    return int(m.group(1)) if m else None

df_f1['num_falta'] = df_f1['Falta o Situacion Tipo I.'].apply(extraer_num_falta)
df_f1['docente_cod'] = df_f1['Autor'].map(docente_map).fillna('otro')

# Agregar F1 por estudiante: conteo total + faltas más frecuentes
f1_agg = df_f1.groupby('Código').agg(
    n_f1=('Id', 'count'),
    falta_mas_frecuente=('num_falta', lambda x: x.mode()[0] if x.notna().any() else None),
    n_tipos_falta=('num_falta', 'nunique'),
).reset_index().rename(columns={'Código': 'codigo_matricula'})

# F2
df_f2 = pd.read_csv(f"{BRONZE}/Consolidado_seguimiento (1).xls",
                    sep=";", encoding="utf-8-sig")
df_f2['docente_cod'] = df_f2['Autor'].map(docente_map).fillna('otro')
f2_agg = df_f2.groupby('Código').agg(
    n_f2=('Id', 'count'),
).reset_index().rename(columns={'Código': 'codigo_matricula'})

print(f"✓ Asistencia: {len(da)} estudiantes")
print(f"✓ F1: {len(f1_agg)} estudiantes con registros")
print(f"✓ F2: {len(f2_agg)} estudiantes con registros")
print(f"\nTop 5 faltas más frecuentes (F1):")
print(df_f1['num_falta'].value_counts().head(5).to_string())

In [0]:
# CELDA 5 — Consolidar, anonimizar y guardar en Trusted (Parquet)

# Join notas + asistencia
da_join = da[['nombre_key', 'codigo_matricula', 'ausencia_clase',
              'ausencia_clase_excusa', 'ausencia_colegio',
              'ausencia_colegio_excusa', 'retardo',
              'salida_temprana', 'total_ausencias']].copy()

df = df_notas.merge(da_join, on='nombre_key', how='left')
df = df.merge(f1_agg, on='codigo_matricula', how='left')
df = df.merge(f2_agg, on='codigo_matricula', how='left')

df['n_f1'] = df['n_f1'].fillna(0).astype(int)
df['n_f2'] = df['n_f2'].fillna(0).astype(int)
df['n_tipos_falta'] = df['n_tipos_falta'].fillna(0).astype(int)

# Anonimización
MODO_ANONIMO = True  # False para uso real en Vermont

if MODO_ANONIMO:
    df['id_estudiante'] = df['codigo_matricula']  # código ya es anónimo
    df['seccion_anon']  = df['curso_orig'].map(seccion_map)
    df.drop(columns=['nombre_key', 'curso_orig'], inplace=True)
else:
    # Modo real: mantener nombre y sección original
    df['id_estudiante'] = df['codigo_matricula']
    df['seccion_anon']  = df['curso_orig']
    df.drop(columns=['nombre_key', 'curso_orig'], inplace=True)

# Reordenar columnas
cols_front = [
    'id_estudiante', 'seccion_anon', 'grado',
    'nivel_riesgo', 'n_asig_bajo_4',
    'promedio_acumulado', 'nota_min_acumulada',
    'ausencia_clase', 'ausencia_clase_excusa',
    'ausencia_colegio', 'ausencia_colegio_excusa',
    'retardo', 'salida_temprana', 'total_ausencias',
    'n_f1', 'n_f2', 'n_tipos_falta', 'falta_mas_frecuente'
]
cols_resto = [c for c in df.columns if c not in cols_front]
df = df[cols_front + cols_resto]

# Guardar como Parquet en Trusted
TRUSTED_PATH = f"{TRUSTED}/estudiantes_2025_26"
spark_df = spark.createDataFrame(df)
spark_df.write.mode("overwrite").parquet(TRUSTED_PATH)

# Verificar
n = spark.read.parquet(TRUSTED_PATH).count()
print(f"✓ Guardado en Trusted: {n} estudiantes")
print(f"  Ruta: {TRUSTED_PATH}")
print(f"  Formato: Parquet")
print(f"  Columnas: {len(df.columns)}")
print(f"  Modo anónimo: {MODO_ANONIMO}")
print(f"\nNulls en columnas clave:")
nulls = df[cols_front].isnull().sum()
print(nulls[nulls > 0].to_string() if nulls[nulls > 0].any() else "  Ninguno ✓")
print(f"\nDistribución riesgo final:")
print(df['nivel_riesgo'].value_counts().to_string())

In [0]:
# Verificación final Trusted
df_check = spark.read.parquet(TRUSTED_PATH).toPandas()

print("=" * 55)
print("RESUMEN ZONA TRUSTED — Vermont School 2025-26")
print("=" * 55)
print(f"Estudiantes: {len(df_check)}")
print(f"Columnas:    {len(df_check.columns)}")
print(f"\nPor grado:")
print(df_check['grado'].value_counts().sort_index().to_string())
print(f"\nPor nivel de riesgo:")
print(df_check['nivel_riesgo'].value_counts().to_string())
print(f"\nEstudiantes con F2: {(df_check['n_f2'] > 0).sum()}")
print(f"Promedio ausencias totales: {df_check['total_ausencias'].mean():.1f}")
print(f"Máximo F1 en un estudiante: {df_check['n_f1'].max()}")
print(f"\nPreview:")
print(df_check[['id_estudiante','seccion_anon','grado',
                'nivel_riesgo','promedio_acumulado',
                'total_ausencias','n_f1','n_f2']].head(5).to_string())